In [3]:
import pandas as pd

In [5]:
df = pd.read_csv('../data/raw/housing_sp_city.csv', encoding='latin1')

## 1. Panorama Inicial do Dataset (Volume de Vendas vs. Locação)

Antes de restringir o escopo do projeto para o modelo preditivo de vendas, realizo uma contagem volumétrica para entender a distribuição dos tipos de anúncios presentes na base de dados bruta de São Paulo.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 133964 entries, 0 to 133963
Data columns (total 18 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   logradouro       126962 non-null  str    
 1   numero           89823 non-null   str    
 2   bairro           133943 non-null  str    
 3   cep              128056 non-null  float64
 4   cidade           133964 non-null  str    
 5   tipo_imovel      133964 non-null  str    
 6   area_util        130868 non-null  float64
 7   banheiros        133051 non-null  float64
 8   suites           120347 non-null  float64
 9   quartos          130945 non-null  float64
 10  vagas_garagem    129539 non-null  float64
 11  anuncio_criado   133964 non-null  str    
 12  tipo_anuncio     133964 non-null  str    
 13  preco_venda      133964 non-null  int64  
 14  taxa_condominio  117127 non-null  float64
 15  periodicidade    29051 non-null   str    
 16  preco_aluguel    28714 non-null   float64
 17  ip

In [7]:
df.head()

,logradouro,numero,bairro,cep,cidade,tipo_imovel,area_util,banheiros,suites,quartos,vagas_garagem,anuncio_criado,tipo_anuncio,preco_venda,taxa_condominio,periodicidade,preco_aluguel,iptu_ano
0,Rua Juvenal Galeno,53,Jardim da SaÃºde,4290030.0,SÃ£o Paulo,Casa de dois andares,388.0,3.0,1.0,4.0,6.0,2017-02-07,Venda,700000,NaN,NaN,NaN,NaN
1,Rua Juruaba,16,Vila Santa Teresa (Zona Sul),4187320.0,SÃ£o Paulo,Casa,129.0,2.0,1.0,3.0,2.0,2016-03-21,Venda,336000,NaN,NaN,NaN,NaN
2,Avenida Paulista,402,Bela Vista,1311000.0,SÃ£o Paulo,Comercial,396.0,4.0,0.0,0.0,5.0,2018-12-18,LocaÃ§Ã£o,24929,4900.0,MONTHLY,29829.0,4040.0
3,Rua Alvorada,1190,Vila OlÃ­mpia,4550004.0,SÃ£o Paulo,Apartamento,80.0,2.0,1.0,3.0,2.0,2018-10-26,Venda,739643,686.0,NaN,NaN,1610.0
4,Rua Curitiba,380,ParaÃ­so,4005030.0,SÃ£o Paulo,Apartamento,3322.0,5.0,4.0,4.0,5.0,2018-12-14,Venda,7520099,6230.0,NaN,NaN,18900.0


In [41]:
# Contabilizo a quantidade absoluta e a proporção percentual de cada tipo de anúncio na base bruta
total_por_tipo = df['tipo_anuncio'].value_counts()
porcentagem_por_tipo = df['tipo_anuncio'].value_counts(normalize=True) * 100

In [42]:
# Exibo os resultados consolidados
for tipo, qtd in total_por_tipo.items():
    pct = porcentagem_por_tipo[tipo]
    print(f"Tipo de Anúncio: {tipo} | Quantidade: {qtd:,} Imóveis | Proporção: {pct:.2f}%")

Tipo de Anúncio: Venda | Quantidade: 105,332 Imóveis | Proporção: 78.63%
Tipo de Anúncio: LocaÃ§Ã£o | Quantidade: 28,632 Imóveis | Proporção: 21.37%


## 2. Filtragem do Escopo e Tratamento de Nulos ($NaN$)

Para garantir que o modelo aprenda a dinâmica correta de precificação de compra e venda, preciso isolar os dados de locação. Durante a análise sensorial dos dados, identifiquei que as colunas `preco_aluguel` e `periodicidade` pertencem ao ecossistema de locação, enquanto as taxas de condomínio nulas refletem imóveis isentos (como casas de rua).

In [18]:
# 1. Filtrando apenas os imovéis a venda
# Como a palavra "locação" está quebrada, vamos uysar a estratégia inversa:
# Mantes apenas onde o tipo_anuncio for exatamente "venda"
df_venda = df[df['tipo_anuncio'] == 'Venda'].copy()

In [19]:
# 2. tratando os NaN do condomínio
df_venda['taxa_condominio'] = df_venda['taxa_condominio'].fillna(0)

In [20]:
total_bairros = df_venda['bairro'].nunique()
print(f'Total de imóveis após filtrar por venda: {df_venda.shape[0]}')
print(f'Quantidade de bairros únicos na cidade de SP: {total_bairros}')

Total de imóveis após filtrar por venda: 105332
Quantidade de bairros únicos na cidade de SP: 1687


In [21]:
df['tipo_anuncio'].unique()

<StringArray>
['Venda', 'LocaÃ§Ã£o']
Length: 2, dtype: str

## 3. Inspeção de Estatísticas Descritivas e Outliers

Com a base de vendas consolidada, analiso as métricas estatísticas das variáveis de preço e área para identificar anomalias nos valores mínimos e máximos antes de iniciar as plotagens gráficas.

In [22]:
# Analiso as estatísticas descritivas das variáveis críticas do modelo
df_venda[['preco_venda', 'area_util']].describe()

,preco_venda,area_util
count,1.053320e+05,1.027630e+05
mean,8.415624e+05,1.972619e+02
std,1.434909e+06,6.477093e+03
min,7.000000e+03,1.000000e+00
25%,2.968000e+05,6.400000e+01
50%,4.822990e+05,1.050000e+02
75%,8.869000e+05,1.820000e+02
max,8.400000e+07,2.025000e+06


## 4. Identificação de Anomalias e Outliers

Ao analisar os extremos dos dados, identifiquei inconsistências severas de cadastro: imóveis com área útil de 1m² ou superiores a 2 milhões de m², além de preços de venda irreais de R$ 7.000. Para facilitar a leitura e o planejamento do corte desses *outliers*, configuro o Pandas para exibir os dados em formato decimal.

In [29]:
# Mudo a configuração do Pandas para exibir floats com duas casas decimais, 
# eleminando a notação cientifica que dificulta a interpretação de valores financeiros
def formatar_metricas(valor):
    return f"{valor:,.2f}"

In [30]:
# Reapresento a descrição estatística para validar a leitura dos extremos
df_venda[['preco_venda', 'area_util']].describe().style.format(formatar_metricas)

,preco_venda,area_util
count,"105,332.00","102,763.00"
mean,"841,562.35",197.26
std,"1,434,908.50","6,477.09"
min,"7,000.00",1.00
25%,"296,800.00",64.00
50%,"482,299.00",105.00
75%,"886,900.00",182.00
max,"84,000,000.00","2,025,000.00"


## 4. Filtragem de Sanidade Baseada em Regras de Negócio

Para eliminar os erros grosseiros de cadastro (como imóveis de 1m² ou valores equivalentes a aluguéis), estabeleço limites mínimos e máximos realistas para o mercado de São Paulo. Defino como corte: área útil entre 15m² e 5.000m², e preço de venda a partir de R$ 80.000, isolando anomalias que prejudicariam o aprendizado dos modelos de regressão.

In [31]:
# Aplicando os limites de corte baseado na análise econômica de mercado
df_limpo = df_venda[
    (df_venda['area_util'] >= 15) &
    (df_venda['area_util'] <= 5000) &
    (df_venda['preco_venda'] >=8000)
].copy()

In [32]:
print(f'Total de imóveis após a limpeza {df_limpo.shape[0]}')

Total de imóveis após a limpeza 102515


## 5. Engenharia de Recursos: Cálculo do Preço por Metro Quadrado ($\text{R\$}/m^2$)

Crio a variável `preco_m2` dividindo o valor total de venda pela área útil. Essa métrica é fundamental para a análise imobiliária, pois normaliza os valores, permitindo identificar distorções remanescentes e servindo como base para avaliar o comportamento econômico dos bairros de alta cardinalidade.

In [34]:
# Criando a nova feature calculando a razão entre o preço e o tamanho do imovel
df_limpo['preco_m2'] = df_limpo['preco_venda'] / df_limpo['area_util']

In [36]:
df_limpo['preco_m2'].describe().to_frame().style.format(formatar_metricas)

,preco_m2
count,"102,515.00"
mean,"5,424.30"
std,"5,398.01"
min,31.16
25%,"3,499.99"
50%,"4,783.33"
75%,"6,695.65"
max,"1,215,789.47"


## 6. Refinamento de Outliers pelo Preço do Metro Quadrado

A criação da métrica `preco_m2` revelou distorções extremas que passaram pelos filtros anteriores, apresentando valores impraticáveis no mercado real (mínimo de R$ 31,16/m² e máximo de R$ 1.215.789,47/m²). Para sanar isso, aplico um novo corte mantendo apenas imóveis cujo valor por metro quadrado esteja entre R$ 2.000 e R$ 50.000, faixas que compreendem desde habitações populares até o altíssimo padrão paulistano.

In [37]:
# Aplico o filtro definitivo baseado no valor do metro quadrado real de SP
df_final = df_limpo[
    (df_limpo['preco_m2'] >= 2000) &
    (df_limpo['preco_m2'] <= 50000)
].copy()


In [38]:
print(f'Total de imóveis após o refino do preço/m²: {df_final.shape[0]}')

Total de imóveis após o refino do preço/m²: 98357


In [39]:
df_final['bairro'].nunique()

1561